# SahamMeter — Spark Analysis (ETS Big Data)
**Kelompok 3 | Bagian D: Spark Analysis**

Notebook ini mencakup:
- Koneksi ke HDFS Oryza (`hdfs://100.74.49.87:8020`)
- Baca data saham (JSON) dari `/data/saham/api/`
- Baca data berita RSS dari `/data/saham/rss/`
- **Analisis 1**: Return per saham
- **Analisis 2**: Volatilitas intraday (stddev)
- **Analisis 3**: Frekuensi sebutan perusahaan di judul berita
- Simpan hasil ke dashboard dan HDFS

## Cell 1 — Import & SparkSession

In [ ]:
import os
import json
import traceback
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.window import Window

# ── Konfigurasi HDFS ──────────────────────────────────────────────────────────
HDFS_HOST      = "100.74.49.87"
HDFS_PORT      = 8020
HDFS_BASE      = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"
HDFS_API_DIR   = f"{HDFS_BASE}/data/saham/api/"
HDFS_RSS_DIR   = f"{HDFS_BASE}/data/saham/rss/"
HDFS_HASIL_DIR = f"{HDFS_BASE}/data/saham/hasil/"

# ── Output lokal ──────────────────────────────────────────────────────────────
DASHBOARD_DIR  = Path("../dashboard/data")
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON    = DASHBOARD_DIR / "spark_results.json"

# ── Perusahaan target untuk Analisis 3 ────────────────────────────────────────
TARGET_COMPANIES = {
    "BCA"    : ["BCA", "Bank Central Asia"],
    "BRI"    : ["BRI", "Bank Rakyat Indonesia"],
    "Telkom" : ["Telkom", "TLKM"],
    "Astra"  : ["Astra", "ASII"],
    "Mandiri": ["Mandiri", "Bank Mandiri"],
}

print("✅ Import selesai")
print(f"📁 Dashboard output : {OUTPUT_JSON.resolve()}")
print(f"🌐 HDFS Base        : {HDFS_BASE}")

## Cell 2 — Inisialisasi SparkSession dengan konfigurasi HDFS

In [ ]:
spark = (
    SparkSession.builder
    .appName("SahamMeter-ETS-Kelompok3")
    # ── HDFS DataNode routing ──────────────────────────────────────────────
    .config("spark.hadoop.dfs.datanode.hostname",          HDFS_HOST)
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    # ── Toleransi & timeout ────────────────────────────────────────────────
    .config("spark.hadoop.fs.hdfs.impl.disable.cache",     "true")
    .config("spark.network.timeout",                        "120s")
    .config("spark.sql.legacy.timeParserPolicy",            "LEGACY")
    # ── JSON multi-line support ────────────────────────────────────────────
    .config("spark.sql.jsonGenerator.ignoreNullFields",     "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Verifikasi
print(f"✅ SparkSession aktif : {spark.version}")
print(f"   App Name          : {spark.sparkContext.appName}")
print(f"   Master            : {spark.sparkContext.master}")

## Cell 3 — Helper: Test koneksi HDFS & list file

In [ ]:
def hdfs_list_files(hdfs_path: str) -> list[str]:
    """List semua file di direktori HDFS. Return list path string."""
    try:
        jvm    = spark._jvm
        conf   = spark._jsc.hadoopConfiguration()
        fs     = jvm.org.apache.hadoop.fs.FileSystem.get(
                     jvm.java.net.URI(hdfs_path), conf)
        status = fs.listStatus(jvm.org.apache.hadoop.fs.Path(hdfs_path))
        paths  = [s.getPath().toString() for s in status]
        return paths
    except Exception as e:
        print(f"⚠️  HDFS list gagal untuk {hdfs_path}: {e}")
        return []


def hdfs_path_exists(hdfs_path: str) -> bool:
    """Cek apakah path HDFS ada."""
    try:
        jvm  = spark._jvm
        conf = spark._jsc.hadoopConfiguration()
        fs   = jvm.org.apache.hadoop.fs.FileSystem.get(
                   jvm.java.net.URI(hdfs_path), conf)
        return fs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))
    except Exception as e:
        print(f"⚠️  HDFS exists check gagal: {e}")
        return False


# ── Test koneksi ───────────────────────────────────────────────────────────────
print("🔍 Test koneksi HDFS...")
for label, path in [("API", HDFS_API_DIR), ("RSS", HDFS_RSS_DIR), ("Hasil", HDFS_HASIL_DIR)]:
    exists = hdfs_path_exists(path)
    files  = hdfs_list_files(path) if exists else []
    status = f"✅ Ada, {len(files)} file" if exists else "❌ Tidak ditemukan"
    print(f"   [{label:5s}] {path}  →  {status}")
    for f in files[:5]:
        print(f"            └─ {f.split('/')[-1]}")
    if len(files) > 5:
        print(f"            └─ ... dan {len(files)-5} file lainnya")

## Cell 4 — Definisi Schema JSON saham

In [ ]:
# Schema fleksibel — sesuaikan dengan struktur JSON aktual dari API
# Jika field berbeda, Spark akan mengisi null secara otomatis

schema_saham = StructType([
    StructField("kode",          StringType(),  True),   # Ticker saham, e.g. "BBCA"
    StructField("nama",          StringType(),  True),   # Nama perusahaan
    StructField("harga_terkini", DoubleType(),  True),   # Harga penutupan / terkini
    StructField("harga_awal",    DoubleType(),  True),   # Harga pembukaan
    StructField("harga_tertinggi", DoubleType(), True),  # High of the day
    StructField("harga_terendah",  DoubleType(), True),  # Low of the day
    StructField("volume",        LongType(),    True),   # Volume transaksi
    StructField("timestamp",     StringType(),  True),   # Waktu data
])

schema_berita = StructType([
    StructField("judul",       StringType(), True),   # Judul berita
    StructField("deskripsi",   StringType(), True),   # Ringkasan berita
    StructField("link",        StringType(), True),   # URL
    StructField("sumber",      StringType(), True),   # Nama media
    StructField("pubDate",     StringType(), True),   # Tanggal publikasi
    StructField("kategori",    StringType(), True),   # Kategori / tag
])

print("✅ Schema didefinisikan")
print("\nSchema Saham:")
schema_saham.printTreeString()
print("\nSchema Berita:")
schema_berita.printTreeString()

## Cell 5 — Baca data saham dari HDFS (dengan fallback dummy)

In [ ]:
def buat_dummy_saham(spark):
    """Data dummy jika HDFS belum ada isinya."""
    print("   ⚠️  Menggunakan data DUMMY untuk saham (HDFS kosong/tidak bisa diakses)")
    dummy = [
        ("BBCA", "Bank Central Asia",     9450.0, 9300.0, 9500.0, 9280.0, 12_500_000, "2025-04-01T09:00:00"),
        ("BBCA", "Bank Central Asia",     9500.0, 9450.0, 9550.0, 9400.0, 10_200_000, "2025-04-01T10:00:00"),
        ("BBRI", "Bank Rakyat Indonesia", 4820.0, 4750.0, 4870.0, 4730.0,  8_700_000, "2025-04-01T09:00:00"),
        ("BBRI", "Bank Rakyat Indonesia", 4780.0, 4820.0, 4830.0, 4760.0,  7_900_000, "2025-04-01T10:00:00"),
        ("TLKM", "Telkom Indonesia",      3910.0, 3880.0, 3950.0, 3870.0,  5_600_000, "2025-04-01T09:00:00"),
        ("TLKM", "Telkom Indonesia",      3930.0, 3910.0, 3960.0, 3900.0,  6_100_000, "2025-04-01T10:00:00"),
        ("ASII", "Astra International",   5250.0, 5200.0, 5300.0, 5180.0,  9_000_000, "2025-04-01T09:00:00"),
        ("ASII", "Astra International",   5300.0, 5250.0, 5350.0, 5230.0,  8_500_000, "2025-04-01T10:00:00"),
        ("BMRI", "Bank Mandiri",          6100.0, 6050.0, 6150.0, 6020.0, 11_000_000, "2025-04-01T09:00:00"),
        ("BMRI", "Bank Mandiri",          6050.0, 6100.0, 6120.0, 6000.0, 10_500_000, "2025-04-01T10:00:00"),
    ]
    return spark.createDataFrame(dummy, schema=schema_saham)


# ── Baca dari HDFS ─────────────────────────────────────────────────────────────
print("📂 Membaca data saham dari HDFS...")
df_saham = None
SAHAM_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_API_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_API_DIR}")

    file_list = hdfs_list_files(HDFS_API_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS API")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_saham = (
        spark.read
        .option("multiLine", "true")   # support JSON array atau multi-line
        .option("mode", "PERMISSIVE")  # abaikan baris rusak, isi null
        .schema(schema_saham)
        .json(HDFS_API_DIR + "*.json")
    )

    count = df_saham.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    SAHAM_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data saham dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_saham = buat_dummy_saham(spark)

# Drop baris yang harga_terkini atau harga_awal null
df_saham = df_saham.dropna(subset=["kode", "harga_terkini", "harga_awal"])

print(f"\n📊 Preview data saham ({'HDFS' if SAHAM_DARI_HDFS else 'DUMMY'}):")
df_saham.printSchema()
df_saham.show(10, truncate=False)

## Cell 6 — Baca data berita RSS dari HDFS (dengan fallback dummy)

In [ ]:
def buat_dummy_berita(spark):
    """Data dummy jika HDFS belum ada isinya."""
    print("   ⚠️  Menggunakan data DUMMY untuk berita (HDFS kosong/tidak bisa diakses)")
    dummy = [
        ("BCA catat laba bersih Rp 48 triliun di kuartal pertama",
         "Bank Central Asia membukukan...", "https://cnbc.id/1", "CNBC Indonesia", "2025-04-01", "perbankan"),
        ("BRI ekspansi kredit UMKM hingga pelosok desa",
         "Bank Rakyat Indonesia...",         "https://cnbc.id/2", "Bisnis.com",      "2025-04-01", "perbankan"),
        ("Telkom luncurkan layanan 5G di 10 kota besar",
         "TLKM mempercepat...",              "https://detik.com/1","Detik Finance",   "2025-04-01", "teknologi"),
        ("Astra International genjot penjualan EV 2025",
         "ASII berencana...",                "https://detik.com/2","Kontan",          "2025-04-01", "otomotif"),
        ("Mandiri syariah berencana IPO tahun ini",
         "Bank Mandiri...",                  "https://tempo.co/1", "Tempo",           "2025-04-01", "perbankan"),
        ("BCA dan Mandiri berebut nasabah digital",
         "Persaingan ketat antara BCA...",   "https://tempo.co/2", "Tempo",           "2025-04-01", "perbankan"),
        ("Telkom dan Astra jalin kerja sama IoT",
         "Dua perusahaan besar ini...",      "https://cnbc.id/3",  "CNBC Indonesia",  "2025-04-02", "teknologi"),
        ("BRI raih penghargaan bank terbaik Asia Tenggara",
         "Penghargaan internasional...",     "https://bisnis.id/1","Bisnis.com",      "2025-04-02", "perbankan"),
        ("IHSG melemah, saham BCA dan Mandiri ikut tertekan",
         "Indeks Harga Saham...",            "https://bisnis.id/2","Bisnis.com",      "2025-04-02", "pasar modal"),
        ("Astra catat penjualan kendaraan 120.000 unit Q1",
         "Penjualan Astra International...","https://kompas.id/1","Kompas",          "2025-04-02", "otomotif"),
    ]
    return spark.createDataFrame(dummy, schema=schema_berita)


# ── Baca dari HDFS ─────────────────────────────────────────────────────────────
print("📰 Membaca data berita dari HDFS...")
df_berita = None
BERITA_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_RSS_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_RSS_DIR}")

    file_list  = hdfs_list_files(HDFS_RSS_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS RSS")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_berita = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema_berita)
        .json(HDFS_RSS_DIR + "*.json")
    )

    count = df_berita.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    BERITA_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data berita dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_berita = buat_dummy_berita(spark)

df_berita = df_berita.dropna(subset=["judul"])

print(f"\n📊 Preview data berita ({'HDFS' if BERITA_DARI_HDFS else 'DUMMY'}):")
df_berita.show(10, truncate=60)

## Cell 7 — Analisis 1: Return per Saham

In [ ]:
# Formula: (harga_terkini - harga_awal) / harga_awal * 100
# Jika ada banyak row per ticker, ambil harga_awal = min(timestamp) dan
# harga_terkini = max(timestamp) supaya return bermakna.

print("━" * 60)
print("📈 ANALISIS 1: Return per Saham")
print("━" * 60)

# Jika df_saham sudah 1 baris per ticker (snapshot), langsung hitung
# Jika banyak snapshot per ticker, aggregate dulu
df_agg = df_saham.groupBy("kode", "nama").agg(
    F.first("harga_awal",    ignorenulls=True).alias("harga_awal"),
    F.last("harga_terkini",  ignorenulls=True).alias("harga_terkini"),
    F.max("harga_tertinggi").alias("harga_tertinggi"),
    F.min("harga_terendah").alias("harga_terendah"),
    F.sum("volume").alias("total_volume"),
)

df_return = df_agg.withColumn(
    "return_pct",
    F.round(
        (F.col("harga_terkini") - F.col("harga_awal")) / F.col("harga_awal") * 100,
        4
    )
).withColumn(
    "status",
    F.when(F.col("return_pct") > 0, "NAIK")
     .when(F.col("return_pct") < 0, "TURUN")
     .otherwise("FLAT")
).orderBy(F.col("return_pct").desc())

print("\nReturn per saham (diurutkan dari tertinggi):")
df_return.select(
    "kode", "nama", "harga_awal", "harga_terkini", "return_pct", "status", "total_volume"
).show(truncate=False)

# Simpan sebagai variabel untuk export
result_return = df_return.select(
    "kode", "nama", "harga_awal", "harga_terkini",
    "harga_tertinggi", "harga_terendah", "return_pct", "status", "total_volume"
).toPandas().to_dict(orient="records")

print(f"✅ Analisis 1 selesai — {len(result_return)} saham diproses")

## Cell 8 — Analisis 2: Volatilitas Intraday (Stddev Harga)

In [ ]:
print("━" * 60)
print("📉 ANALISIS 2: Volatilitas Intraday per Saham")
print("━" * 60)

# Volatilitas dihitung dari stddev harga_terkini antar snapshot
# Jika hanya 1 snapshot per saham, juga hitung range (high - low)
df_volatilitas = df_saham.groupBy("kode", "nama").agg(
    F.round(F.stddev("harga_terkini"), 4).alias("stddev_harga"),
    F.round(F.avg("harga_terkini"), 4).alias("rata_rata_harga"),
    F.max("harga_tertinggi").alias("high"),
    F.min("harga_terendah").alias("low"),
    F.count("*").alias("jumlah_snapshot"),
).withColumn(
    # Range intraday sebagai ukuran volatilitas alternatif
    "range_intraday",
    F.round(F.col("high") - F.col("low"), 2)
).withColumn(
    # Coefficient of variation (stddev/mean * 100) — jika ada >1 snapshot
    "cv_pct",
    F.round(
        F.when(F.col("rata_rata_harga") > 0,
               F.col("stddev_harga") / F.col("rata_rata_harga") * 100)
         .otherwise(0.0),
        4
    )
).withColumn(
    "level_volatilitas",
    F.when(F.col("cv_pct") > 2.0, "TINGGI")
     .when(F.col("cv_pct") > 0.5, "SEDANG")
     .otherwise("RENDAH")
).orderBy(F.col("cv_pct").desc())

print("\nVolatilitas intraday (diurutkan dari tertinggi):")
df_volatilitas.show(truncate=False)

print("\nCatatan:")
print("  • stddev_harga    : simpangan baku harga antar snapshot")
print("  • range_intraday  : high - low hari ini")
print("  • cv_pct          : coefficient of variation (stddev/mean × 100)")
print("  • NULL stddev      normal jika hanya 1 snapshot per saham")

result_volatilitas = df_volatilitas.toPandas().fillna(0).to_dict(orient="records")
print(f"\n✅ Analisis 2 selesai — {len(result_volatilitas)} saham diproses")

## Cell 9 — Analisis 3: Frekuensi Sebutan Perusahaan di Judul Berita

In [ ]:
print("━" * 60)
print("📰 ANALISIS 3: Frekuensi Sebutan Perusahaan di Berita")
print("━" * 60)
print(f"   Target: {list(TARGET_COMPANIES.keys())}")

# Tambahkan kolom boolean per perusahaan berdasarkan keyword di judul
df_tagged = df_berita

for company, keywords in TARGET_COMPANIES.items():
    # Bangun regex OR dari semua keyword
    pattern = "|".join(keywords)
    col_name = f"mention_{company.lower()}"
    df_tagged = df_tagged.withColumn(
        col_name,
        F.when(
            F.regexp_extract(F.upper(F.col("judul")), pattern.upper(), 0) != "",
            1
        ).otherwise(0)
    )

print("\nSample tagging berita:")
df_tagged.select(
    "judul",
    *[f"mention_{c.lower()}" for c in TARGET_COMPANIES]
).show(10, truncate=55)

# Agregasi: hitung total mention per perusahaan
mention_counts = {}
for company in TARGET_COMPANIES:
    col_name = f"mention_{company.lower()}"
    count = df_tagged.agg(F.sum(col_name).cast("long")).collect()[0][0] or 0
    mention_counts[company] = int(count)

# Buat DataFrame hasil
rows_frekuensi = [
    (company, int(count), list(TARGET_COMPANIES[company]))
    for company, count in sorted(mention_counts.items(), key=lambda x: -x[1])
]

schema_frekuensi = StructType([
    StructField("perusahaan",   StringType(), True),
    StructField("jumlah_sebutan", LongType(), True),
])

df_frekuensi = spark.createDataFrame(
    [(r[0], r[1]) for r in rows_frekuensi],
    schema=schema_frekuensi
)

print("\n🏆 Peringkat sebutan perusahaan di berita:")
df_frekuensi.show()

result_frekuensi = [{"perusahaan": r[0], "jumlah_sebutan": r[1], "keywords": r[2]}
                    for r in rows_frekuensi]

print(f"✅ Analisis 3 selesai")

## Cell 10 — Gabungkan semua hasil & simpan ke dashboard JSON

In [ ]:
print("💾 Menyimpan hasil analisis ke dashboard...")

output_payload = {
    "metadata": {
        "project"       : "SahamMeter",
        "kelompok"      : 3,
        "generated_at"  : datetime.now().isoformat(),
        "spark_version" : spark.version,
        "saham_dari_hdfs" : SAHAM_DARI_HDFS,
        "berita_dari_hdfs": BERITA_DARI_HDFS,
        "hdfs_base"     : HDFS_BASE,
    },
    "analisis_1_return": result_return,
    "analisis_2_volatilitas": result_volatilitas,
    "analisis_3_frekuensi_berita": result_frekuensi,
}

# Simpan ke file lokal
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2, default=str)

size_kb = OUTPUT_JSON.stat().st_size / 1024
print(f"   ✅ Tersimpan ke : {OUTPUT_JSON.resolve()}")
print(f"   📦 Ukuran file  : {size_kb:.2f} KB")
print(f"   🕐 Timestamp    : {output_payload['metadata']['generated_at']}")

## Cell 11 — Simpan hasil ke HDFS /data/saham/hasil/

In [ ]:
print("☁️  Menyimpan hasil ke HDFS...")

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

def simpan_ke_hdfs(df, hdfs_path: str, label: str):
    """Simpan DataFrame ke HDFS sebagai JSON. Return True jika sukses."""
    try:
        (
            df.coalesce(1)            # 1 file output
            .write
            .mode("overwrite")
            .option("encoding", "UTF-8")
            .json(hdfs_path)
        )
        print(f"   ✅ [{label}] tersimpan ke {hdfs_path}")
        return True
    except Exception as e:
        print(f"   ❌ [{label}] gagal simpan ke HDFS: {e}")
        return False


# Definisi output HDFS per analisis
hdfs_outputs = [
    (df_return,      f"{HDFS_HASIL_DIR}return_{timestamp_str}",      "Return Saham"),
    (df_volatilitas, f"{HDFS_HASIL_DIR}volatilitas_{timestamp_str}", "Volatilitas"),
    (df_frekuensi,   f"{HDFS_HASIL_DIR}frekuensi_{timestamp_str}",   "Frekuensi Berita"),
]

hdfs_sukses = 0
for df, path, label in hdfs_outputs:
    ok = simpan_ke_hdfs(df, path, label)
    if ok:
        hdfs_sukses += 1

print(f"\n   Total tersimpan ke HDFS: {hdfs_sukses}/{len(hdfs_outputs)}")

if hdfs_sukses == 0:
    print("   ⚠️  Semua gagal ke HDFS — hasil tetap tersimpan di dashboard lokal")

## Cell 12 — Ringkasan Akhir & Validasi

In [ ]:
print("═" * 60)
print("  RINGKASAN HASIL — SahamMeter ETS Big Data Kelompok 3")
print("═" * 60)

print(f"\n📌 Sumber Data:")
print(f"   Saham  : {'HDFS ✅' if SAHAM_DARI_HDFS else 'DUMMY ⚠️'}")
print(f"   Berita : {'HDFS ✅' if BERITA_DARI_HDFS else 'DUMMY ⚠️'}")

print(f"\n📊 Analisis 1 — Return per Saham:")
for r in result_return:
    arrow = "▲" if r["return_pct"] > 0 else ("▼" if r["return_pct"] < 0 else "─")
    print(f"   {r['kode']:6s} {arrow} {r['return_pct']:+.2f}%  "
          f"(Rp{r['harga_awal']:,.0f} → Rp{r['harga_terkini']:,.0f})")

print(f"\n📊 Analisis 2 — Volatilitas Intraday:")
for r in result_volatilitas:
    print(f"   {r['kode']:6s}  stddev={r['stddev_harga']:.2f}  "
          f"range={r['range_intraday']:.0f}  [{r['level_volatilitas']}]")

print(f"\n📊 Analisis 3 — Frekuensi Sebutan Berita:")
for r in result_frekuensi:
    bar = "█" * r["jumlah_sebutan"]
    print(f"   {r['perusahaan']:8s} {bar} {r['jumlah_sebutan']}×")

print(f"\n💾 Output:")
print(f"   Lokal  : {OUTPUT_JSON.resolve()}")
print(f"   HDFS   : {HDFS_HASIL_DIR}*_{timestamp_str}/")
print(f"\n✅ Semua analisis selesai — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("═" * 60)

## Cell 13 — Stop SparkSession

In [ ]:
spark.stop()
print("🛑 SparkSession dihentikan")